# GEP-OnSSET GIS-Extraction Notebook for GEP-OnSSET

This is the GIS extraction notebook. The main purpose of this notebook is to extract geospatial data required for the least-cost electrification analysis using the OnSSET tool for every settlement in the country.

In order to run an OnSSET analysis the following datasets are strictly required:
* Admin (country) boundaries
* Global horizontal irradiation
* Travel time
* Wind velocity
* Clusters **(these clusters should include: The name of the study area, the amount of nighttime lights, population, population living in areas with nighttime light and an ID column)**. The clusters can be downloaded from the SDI or generated directly using this [code](https://github.com/Mozambique-IEP/OnSSET-Clustering).

In addition to this there are also some optional datasets that can be used to improve the analysis:
* Substations
* Distribution transformers
* Existing mini-grids
* Admin 1 (province) boundary layers
* Mini/Small hydro potential
* Existing and planned HV-lines
* Existing and planned MV-lines 
* Road network

Below instructions for each cell follows. The cells marked with **(Mandatory)** in the title have to be run.

### Useful hints and common error messages
* Make sure that all input layers are using EPSG:4326 as the coordinate system
* Make sure that the "crs" in cell 2 is in a coordinate system using meters as the unit
* It is often useful to clip all the input layers to the country boundaries in order to reduce processing times
* Make sure that each dataset actually has some data within the country boundaries
* Some of the datasets require the user to choose values from a dropdown list below
* For hydro points and mini-grids, the vector layers need some specific column names to work
* In case a dataset still does not work, try opening it in QGIS and run the *Fix geometries* tool and save the new layer.
* If things do not work, it may be useful to go to the very top of this Jupyter Notebook and start again from cell 1


## Step 1 - Importing necessary packages (Mandatory)

Packages to be used are imported from the funcs.ipynb.

In [ ]:
import time
import sys
from pathlib import Path
root_dir = Path.cwd().parent
sys.path.insert(0, str(root_dir))
from scripts.funcs import *
workspace = os.path.join(root_dir, 'data', 'outputs')

## Step 2 - Setting the target coordinate system (Mandatory)

When calculating distances it is important to choose a coordinate system that represents distances correctly in your area of interst. The coordinate system that is given below is the World Mercator, these coordinate system generally work well, but the distortions get larger as you move away from the equator.

In order to select your own coordinate system go to [epsg.io](http://epsg.io/) and type in your area of interest, this will give you a list of coordinate systems to choose from. Once you have selected your coordinate system replace the numbers below with the numbers from your coordinate system **(keep the "EPSG" part)**.

**NOTE** When selecting your coordinate system make sure that you select a system with the unit of meters, this is indicated for all systems on [epsg.io](http://epsg.io/)

In [ ]:
crs = 'EPSG:3395'

## Step 3 - Select the administrative boundaries (Mandatory)

For the administrative boundaries you will have to select an **Polygon** layer represeting your area of interest.
        

In [ ]:
messagebox.showinfo('OnSSET', 'Select the admin boundaries')
admin_path = filedialog.askopenfilename(initialdir=os.path.join(root_dir, 'data', 'inputs', 'AdminBoundaries'),
                                        filetypes = (("vector",["*.shp", "*.gpkg", "*.geojson", "*.parquet"]),("all files","*.*")))
admin = gpd.read_file(admin_path)

## Step 4 - Select the population clusters (Mandatory)

Select the clusters to be used in the analysis

Please also indicate which column is representing the population data as this will be used later. 

In [ ]:
x, clusters, clusters_path = select_pop_clusters(os.path.join(root_dir, 'data', 'inputs', 'Clusters'))
clusters = clusters[~clusters.geometry.is_empty & clusters.geometry.notnull()].copy()
# Fix invalid geometries
clusters['geometry'] = clusters['geometry'].buffer(0)

## Step 5 - Select the Global Horizontal Irradiation (GHI) map - Raster map (Mandatory)

**If your settlement data already includes GHI data, skip to the next step. Note however that this dataset is mandatory to run the OnSSET analysis**

Select the ghi map that you wish to use in your analysis. This cell will extract the ghi values in your raster map to your clusters.

In [ ]:
out, ghi_path = zonal_stats_exact('GHI', clusters, 'mean', init_dir=os.path.join(root_dir, 'data', 'inputs', 'SolarGHI'))
if type(out) == gpd.geodataframe.GeoDataFrame:
    clusters = out

## Step 6 - Select the Travel Time map - Raster map (Mandatory)
 
**If your settlement data already includes travel time data, skip to the next step. Note however that this dataset is mandatory to run the OnSSET analysis**

Select the travel time map that you wish to use in your analysis. This cell will extract the travel time values in your raster map to your clusters.

In [ ]:
out, traveltime_path = zonal_stats_exact('TravelTime', clusters, 'mean', init_dir=os.path.join(root_dir, 'data', 'inputs', 'TravelTime'))
if type(out) == gpd.geodataframe.GeoDataFrame:
    clusters = out

## Step 7 - Select the Wind Velocity map - Raster map (Mandatory)

**If your settlement data already includes wind velocity data, skip to the next step. Note however that this dataset is mandatory to run the OnSSET analysis**

Select the wind velocity map that you wish to use in your analysis. This cell will extract the wind velocity values in your raster map to your clusters.

In [ ]:
out, wind_path = zonal_stats_exact('WindVel', clusters, 'mean', init_dir=os.path.join(root_dir, 'data', 'inputs', 'WindVelocity'))
if type(out) == gpd.geodataframe.GeoDataFrame:
    clusters = out

## Step 8 - Select the Night Lights map - Raster map (Mandatory)

**If your settlement data already includes night lights data, skip to teh next step. Note however that this dataset is mandatory to run the OnSSET analysis**

Select the wind velocity map that you wish to use in your analysis. This cell will extract the wind velocity values in your raster map to your clusters.

In [ ]:
out, ntl_path = zonal_stats_exact('NightLight', clusters, 'max', init_dir=os.path.join(root_dir, 'data', 'inputs', 'NightTimeLights'))
if type(out) == gpd.geodataframe.GeoDataFrame:
    clusters = out

## Step 9 - Preparing to run the vector data (Mandatory)

**If you are planning on extracting any vector data (substations, transformers, hydro, MV-lines, HV-lines or roads) run this cell**. 

This cell reprojects the settlements to the coordinate system you specified above.

In [ ]:
clusters = preparing_for_vectors(workspace, clusters, crs)

## Step 10 - Substations - Vector point layer (Optional)

**If you do not have substations or wish to keep the ones already in your settlement file, skip to the next step.**

Determines the distances between each settlement point to the closest substation. 

In [ ]:
out, substation_path = processing_points("Substation", admin, crs, workspace, clusters, mg_filter=False, init_dir=os.path.join(root_dir, 'data', 'inputs', 'Substations'))
if type(out) == gpd.geodataframe.GeoDataFrame:
    clusters = out

## Step 11 - Existing high voltage lines - Vector line layer (Optional)

**If you do not have existing high voltage lines or wish to keep the ones already in your settlement file, skip to the next step.**

Determines the distances between each settlement point to the closest existing high voltage line. 

In [ ]:
out, existing_hv_path = processing_lines("Existing_HV", admin, crs, workspace, clusters, init_dir=os.path.join(root_dir, 'data', 'inputs', 'HV_Existing'))
if type(out) == gpd.geodataframe.GeoDataFrame:
    clusters = out

## Step 12 - Planned high voltage lines - Vector line layer (Optional)

**If you do not have planned high voltage lines or wish to keep the ones already in your settlement file, skip to the next step.**

Determines the distances between each settlement point to the closest planned high voltage line. 

In [ ]:
out, planned_hv_path = processing_lines("Planned_HV", admin, crs, workspace, clusters, init_dir=os.path.join(root_dir, 'data', 'inputs', 'HV_Planned'))
if type(out) == gpd.geodataframe.GeoDataFrame:
    clusters = out

## Step 13 - Existing medium voltage lines - Vector line layer (Optional)

**If you do not have existing medium voltage lines or wish to keep the ones already in your settlement file, skip to the next step.**

Determines the distances between each settlement point to the closest existing medium voltage line. 

In [ ]:
out, existing_mv_path = processing_lines("Existing_MV", admin, crs, workspace, clusters, init_dir=os.path.join(root_dir, 'data', 'inputs', 'MV_Existing'))
if type(out) == gpd.geodataframe.GeoDataFrame:
    clusters = out

## Step 14 - Planned medium voltage lines - Vector line layer (Optional)

**If you do not have planned medium voltage lines or wish to keep the ones already in your settlement file, skip to the next step.**

Determines the distances between each settlement point to the closest planned medium voltage line. 

In [ ]:
out, planned_mv_path = processing_lines("Planned_MV", admin, crs, workspace, clusters, init_dir=os.path.join(root_dir, 'data', 'inputs', 'MV_Planned'))
if type(out) == gpd.geodataframe.GeoDataFrame:
    clusters = out

## Step 15 - Roads - Vector line layer (Optional)

**If you do not have roads or wish to keep the ones already in your settlement file, skip to the next step.**

Determines the distances between each settlement point to the closest road. 

In [ ]:
out, roads_path = processing_lines("Road", admin, crs, workspace, clusters, init_dir=os.path.join(root_dir, 'data', 'inputs', 'Roads'))
if type(out) == gpd.geodataframe.GeoDataFrame:
    clusters = out

## Step 16 - Service/distribution transformers - Vector point layer (Optional)

**If you do not have transformers or wish to keep the ones in the already in the settlement file, skip to the next step.** 

Determines the distances between each settlement point to the closest transformer. 

In [ ]:
out, service_transformer_path = processing_points("Service Transformer", admin, crs, workspace, clusters, mg_filter=False, init_dir=os.path.join(root_dir, 'data', 'inputs', 'DistributionTransformers'))
if type(out) == gpd.geodataframe.GeoDataFrame:
    clusters = out

## Step 17 - Hydro points - Vector point layer (Optional)

**If you do not have new hydro power points skip to next step**

Select the hydro point layer you wish to use. It is important to have a column representing the power output for each hydro point in your dataset. After selecting the column you will also have to select the unit (W, kW or MW) of that column. 

In [ ]:
out, hydro_path = hydro(admin, crs, workspace, clusters, init_dir=os.path.join(root_dir, 'data', 'inputs', 'HydroPotential'))
if type(out) == gpd.geodataframe.GeoDataFrame:
    clusters = out

# Extra datasets that can be used to improve the analysis

## Step 18 - Existing mini-grids - Vector point layer (Optional extra)

This function extracts the nearest mini-grid to each clusters and assigns key characteristics (e.g. name, MV network status, type).

In [ ]:
out, mg_path = processing_points("MiniGrid", admin, crs, workspace, clusters, mg_filter=True, init_dir=os.path.join(root_dir, 'data', 'inputs', 'MiniGrids'))
if type(out) == gpd.geodataframe.GeoDataFrame:
    clusters = out

## Step 19 - Extracting admin 1 name to clusters - Vector polygon layer (Optional extra)

This function extracts the admin level 1 (e.g. Province, State) name to each cluster based on spatial overlay. 

**Please do provide the right column name (e.g. "adm1_name") in the pop-up window**

In [ ]:
out, admin_1_path = admin_1(clusters, init_dir=os.path.join(root_dir, 'data', 'inputs', 'AdminBoundaries'))
if type(out) == gpd.geodataframe.GeoDataFrame:
    clusters = out

## Step 20 - Conditioning & Export (Mandatory)

This is the final step in the extraction. This cell has to be run.

**Note that this will overwrite any existing input file in the data/outputs folder**

In [ ]:
clusters = conditioning(clusters, workspace, x)
print('Workspace: ', workspace)